# HVM Sweep Results
Loads all `checkpoints/hvm/*/siglip/*/best.pt` and compares performance across:
- Model architecture (MLP / LSTM / Transformer)
- Category conditioning (with vs without)
- Hyperparameters (bottleneck/hidden/d_model, lr, weight_decay)

Primary metric: **val 2-AFC identification**. Secondary: val/test cosine similarity.

In [ ]:
import sys
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

sys.path.insert(0, '..')
from config_const import CHECKPOINT_DIR

HVM_CKPT_ROOT = CHECKPOINT_DIR / 'hvm'

def load_one(pt):
    try:
        ckpt = torch.load(pt, weights_only=False, map_location='cpu')
    except Exception as e:
        return None
    if 'metrics' not in ckpt:
        return None
    row = {**ckpt['args'], **ckpt['metrics'], 'path': str(pt)}
    if 'test_metrics' in ckpt:
        row.update(ckpt['test_metrics'])
    return row

candidates = sorted(HVM_CKPT_ROOT.glob('*/siglip/*/best.pt'))
print(f'Found {len(candidates)} checkpoints')

records = []
with ThreadPoolExecutor(max_workers=8) as pool:
    futures = {pool.submit(load_one, pt): pt for pt in candidates}
    for fut in tqdm(as_completed(futures), total=len(futures), desc='Loading'):
        row = fut.result()
        if row:
            records.append(row)

df = pd.DataFrame(records)
has_test = 'test_2afc' in df.columns

# Normalise use_category to bool
if 'use_category' in df.columns:
    df['use_category'] = df['use_category'].astype(bool)
else:
    df['use_category'] = False

print(f'{len(df)} runs loaded  |  test_metrics: {has_test}')
print(df.groupby(['model', 'use_category']).size().to_string())

In [ ]:
def make_arch(row):
    m = row['model']
    if m == 'mlp':
        return f"bn{int(row['bottleneck'])}"
    elif m == 'lstm':
        return f"h{int(row['hidden'])}_nl{int(row['n_layers'])}"
    else:
        return f"d{int(row['d_model'])}_nl{int(row['n_layers'])}"

df['arch'] = df.apply(make_arch, axis=1)
df['cat_label'] = df['use_category'].map({True: 'cat', False: 'no_cat'})

MODELS = sorted(df['model'].unique())
MODEL_PALETTE = dict(zip(MODELS, ['tab:blue', 'tab:orange', 'tab:green']))
CAT_STYLE = {True: dict(marker='o', ls='-', label='conditioned'), 
             False: dict(marker='s', ls='--', label='unconditioned')}
print(f'Models: {MODELS}')

## Best run per model × conditioning

In [ ]:
cols = ['model', 'use_category', 'arch', 'dropout', 'lr', 'weight_decay',
        'epoch', 'train_loss', 'val_loss', 'val_2afc', 'val_cos_sim',
        'test_2afc', 'test_top1', 'test_top5', 'test_top10']
available = [c for c in cols if c in df.columns]

summary = (
    df.sort_values('val_2afc', ascending=False)
      .groupby(['model', 'use_category'], sort=False)
      .first()
      .reset_index()
)[available]
display(summary)

## Category conditioning: lift in 2-AFC and cosine similarity

In [ ]:
metrics = [('val_2afc', 'Val 2-AFC'), ('val_cos_sim', 'Val cos sim')]
if has_test:
    metrics += [('test_2afc', 'Test 2-AFC'), ('test_cos_sim', 'Test cos sim')]
metrics = [(m, l) for m, l in metrics if m in df.columns]

fig, axes = plt.subplots(1, len(metrics), figsize=(4.5 * len(metrics), 4.5))
if len(metrics) == 1:
    axes = [axes]

x = np.arange(len(MODELS))
width = 0.35

for ax, (metric, label) in zip(axes, metrics):
    best = (
        df.sort_values(metric, ascending=False)
          .groupby(['model', 'use_category'])
          .first()
          .reset_index()
    )
    for i, use_cat in enumerate([False, True]):
        vals = [best.loc[(best['model'] == m) & (best['use_category'] == use_cat), metric]
                     .values for m in MODELS]
        heights = [float(v[0]) if len(v) > 0 else 0.0 for v in vals]
        offset = (i - 0.5) * width
        bars = ax.bar(x + offset, heights, width,
                      label='with cat' if use_cat else 'no cat',
                      color=['tab:blue' if use_cat else 'tab:gray'],
                      alpha=0.85)
        for bar, h in zip(bars, heights):
            if h > 0:
                ax.text(bar.get_x() + bar.get_width() / 2, h + 0.005,
                        f'{h:.3f}', ha='center', va='bottom', fontsize=7, rotation=90)
    if 'afc' in metric:
        ax.axhline(0.5, color='grey', lw=1, ls='--', label='chance')
    ax.set_xticks(x)
    ax.set_xticklabels(MODELS)
    ax.set_title(label)
    ax.legend(fontsize=8)

fig.suptitle('Category conditioning: best run per model  (HVM → SigLIP)', fontweight='bold')
plt.tight_layout()
plt.show()

## All runs: val 2-AFC distribution by model and conditioning

In [ ]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(4.5 * len(MODELS), 4.5), sharey=True)
rng = np.random.default_rng(42)

for ax, model in zip(axes, MODELS):
    sub = df[df['model'] == model]
    for use_cat in [False, True]:
        grp = sub[sub['use_category'] == use_cat]['val_2afc'].dropna().values
        if not len(grp):
            continue
        x_pos = 1 if use_cat else 0
        jitter = rng.uniform(-0.12, 0.12, len(grp))
        color = 'tab:blue' if use_cat else 'tab:gray'
        ax.scatter(np.full(len(grp), x_pos) + jitter, grp,
                   color=color, alpha=0.7, s=50)
        ax.plot([x_pos - 0.25, x_pos + 0.25], [grp.mean()] * 2, 'k-', lw=2)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['uncond', 'cat-cond'])
    ax.set_title(model, fontweight='bold')
    ax.axhline(0.5, color='grey', lw=1, ls='--')
    if ax is axes[0]:
        ax.set_ylabel('Val 2-AFC')

fig.suptitle('All runs — val 2-AFC by model and category conditioning', fontweight='bold')
plt.tight_layout()
plt.show()

## Hyperparameter effects — 2-AFC by lr and weight_decay

In [ ]:
def heatmap(ax, grp, row_col, col_col, val_col, title, fmt='.3f'):
    pivot = grp.groupby([row_col, col_col])[val_col].mean().unstack()
    if pivot.empty:
        ax.set_visible(False)
        return
    im = ax.imshow(pivot.values, cmap='viridis', aspect='auto')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f'{v:.0e}' for v in pivot.columns], rotation=45, ha='right')
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f'{v:.0e}' for v in pivot.index])
    ax.set_xlabel(col_col)
    ax.set_ylabel(row_col)
    ax.set_title(title, fontsize=9)
    mean_val = np.nanmean(pivot.values)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            v = pivot.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, format(v, fmt), ha='center', va='center', fontsize=8,
                        color='white' if v < mean_val else 'black')
    plt.colorbar(im, ax=ax)

n_cat_vals = [False, True]
fig, axes = plt.subplots(len(MODELS), 2, figsize=(10, 4.5 * len(MODELS)), squeeze=False)

for ri, model in enumerate(MODELS):
    for ci, use_cat in enumerate(n_cat_vals):
        sub = df[(df['model'] == model) & (df['use_category'] == use_cat)].dropna(subset=['val_2afc'])
        title = f"{model} {'(cat)' if use_cat else '(no cat)'}\nLR × WD → val 2-AFC"
        if {'lr', 'weight_decay'}.issubset(sub.columns) and len(sub):
            heatmap(axes[ri, ci], sub, 'lr', 'weight_decay', 'val_2afc', title)
        else:
            axes[ri, ci].text(0.5, 0.5, 'no data', ha='center', va='center',
                              transform=axes[ri, ci].transAxes)
            axes[ri, ci].set_title(title, fontsize=9)

fig.suptitle('Val 2-AFC: LR × weight_decay heatmap (mean across arch variants)', fontweight='bold')
plt.tight_layout()
plt.show()

## Architecture size effect on 2-AFC

In [ ]:
ARCH_PARAM = {'mlp': 'bottleneck', 'lstm': 'hidden', 'transformer': 'd_model'}

fig, axes = plt.subplots(1, len(MODELS), figsize=(4.5 * len(MODELS), 4.5), sharey=True)
rng2 = np.random.default_rng(0)

for ax, model in zip(axes, MODELS):
    sub = df[df['model'] == model].dropna(subset=['val_2afc'])
    param = ARCH_PARAM[model]
    if param not in sub.columns:
        ax.set_title(f'{model}\n(no arch param)'); continue
    levels = sorted(sub[param].unique())
    for li, lvl in enumerate(levels):
        for use_cat in [False, True]:
            vals = sub[(sub[param] == lvl) & (sub['use_category'] == use_cat)]['val_2afc'].values
            if not len(vals): continue
            x_pos = li + (0.15 if use_cat else -0.15)
            color = 'tab:blue' if use_cat else 'tab:gray'
            jitter = rng2.uniform(-0.08, 0.08, len(vals))
            ax.scatter(np.full(len(vals), x_pos) + jitter, vals,
                       color=color, alpha=0.7, s=45,
                       label=('cat' if use_cat else 'no cat') if li == 0 else '_')
            ax.plot([x_pos - 0.1, x_pos + 0.1], [vals.mean()] * 2, color=color, lw=2)
    ax.set_xticks(range(len(levels)))
    ax.set_xticklabels([str(int(l)) for l in levels])
    ax.set_xlabel(param)
    ax.set_title(model, fontweight='bold')
    ax.axhline(0.5, color='grey', lw=1, ls='--')
    ax.legend(fontsize=8)
    if ax is axes[0]:
        ax.set_ylabel('Val 2-AFC')

fig.suptitle('Architecture size vs val 2-AFC  (blue=cat-conditioned, gray=unconditioned)', fontweight='bold')
plt.tight_layout()
plt.show()

## Val vs Test generalisation

In [ ]:
if has_test:
    fig, axes = plt.subplots(1, len(MODELS), figsize=(4.5 * len(MODELS), 4.5), sharey=True)
    for ax, model in zip(axes, MODELS):
        sub = df[(df['model'] == model)].dropna(subset=['val_2afc', 'test_2afc'])
        for use_cat in [False, True]:
            grp = sub[sub['use_category'] == use_cat]
            if grp.empty: continue
            color = 'tab:blue' if use_cat else 'tab:gray'
            ax.scatter(grp['val_2afc'], grp['test_2afc'], color=color, alpha=0.75, s=55,
                       label='cat' if use_cat else 'no cat')
        if not sub.empty:
            lim = [min(sub['val_2afc'].min(), sub['test_2afc'].min()) - 0.02,
                   max(sub['val_2afc'].max(), sub['test_2afc'].max()) + 0.02]
            ax.plot(lim, lim, 'k--', lw=1, alpha=0.4)
        ax.axhline(0.5, color='grey', lw=1, ls=':', alpha=0.5)
        ax.axvline(0.5, color='grey', lw=1, ls=':', alpha=0.5)
        ax.set_xlabel('Val 2-AFC')
        ax.set_title(model, fontweight='bold')
        ax.legend(fontsize=8)
        if ax is axes[0]:
            ax.set_ylabel('Test 2-AFC')
    fig.suptitle('Val vs Test 2-AFC generalisation  (dashed = y=x)', fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Test metrics not yet available.')

## Top-10 runs overall

In [ ]:
top_cols = [c for c in [
    'model', 'use_category', 'arch', 'lr', 'weight_decay',
    'epoch', 'val_loss', 'val_2afc', 'val_cos_sim',
    'test_2afc', 'test_top1', 'test_top5', 'test_cos_sim', 'path'
] if c in df.columns]
display(df.sort_values('val_2afc', ascending=False)[top_cols].head(10).reset_index(drop=True))

## Save best HVM encoder config

In [ ]:
import json
from config_const import CACHE_DIR

configs = {}
for use_cat in [False, True]:
    sub = df[df['use_category'] == use_cat]
    if sub.empty:
        continue
    best = sub.sort_values('val_2afc', ascending=False).iloc[0]
    key = 'hvm_siglip_cat' if use_cat else 'hvm_siglip'
    cfg = {
        'dataset':       'hvm',
        'target':        'siglip',
        'use_category':  bool(use_cat),
        'model':         best['model'],
        'val_2afc':      float(best['val_2afc']),
        'checkpoint':    best['path'],
    }
    for k in ['bottleneck', 'hidden', 'n_layers', 'd_model', 'n_heads',
              'shared_dim', 'dropout', 'lr', 'weight_decay']:
        if k in best and pd.notna(best[k]):
            cfg[k] = float(best[k]) if isinstance(best[k], float) else int(best[k])
    configs[key] = cfg

out_path = CACHE_DIR / 'best_hvm_encoder_config.json'
with open(out_path, 'w') as f:
    json.dump(configs, f, indent=2)
print(f'Saved to {out_path}')
for k, cfg in configs.items():
    print(f"  {k}: {cfg['model']} arch — val_2afc={cfg['val_2afc']:.4f}")
    print(f"    {cfg['checkpoint']}")